# Multi-Agent Research Assistant with CrewAI

This notebook demonstrates building a **multi-agent research assistant** using CrewAI. Two agents collaborate sequentially:
1. A **Market Researcher** gathers insights from the web using the Serper search tool
2. A **Product Strategist** creates a positioning strategy based on those insights

## What You'll Learn

- How to build a **multi-agent system** where agents have distinct roles
- How to equip agents with **tools** (e.g., `SerperDevTool` for web search)
- How tasks flow from one agent to the next in a **sequential pipeline**
- How CrewAI's `planning=True` mode orchestrates the overall workflow

## Prerequisites

- `OPENAI_API_KEY` set in your environment
- `SERPER_API_KEY` set in your environment (get one free at [serper.dev](https://serper.dev))

## Step 1: Import Dependencies

We import CrewAI's core classes along with `SerperDevTool` from `crewai_tools`, which gives agents the ability to search the web via the Serper API.

In [ ]:
from crewai import Agent, Task, Crew, LLM
import os
from crewai_tools import SerperDevTool  # Web search tool powered by serper.dev
from dotenv import load_dotenv
load_dotenv()

In [ ]:
llm = LLM(
    model="openai/gpt-oss-20b",
    provider="openai",
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ["GROQ_API_KEY"],
    temperature=0,
)

## Step 2: Configure the Research Target

Define the product to research and any additional context for the strategist agent. These variables are injected into task descriptions via f-strings.

In [ ]:
product_name = "energy drink"                     # The product to research and strategize for
strategist_backstory = "and marketing strategy"    # Additional expertise area for the strategist

## Step 3: Define the Agents

We create two specialized agents:

| Agent | Role | Superpower |
|-------|------|-----------|
| **Market Researcher** | Gathers real-time market data | Has access to `SerperDevTool` for web search |
| **Product Strategist** | Turns insights into strategy | Relies on the researcher's output (no tools needed) |

Notice that only the researcher has tools — the strategist works purely from the context it receives.

In [ ]:
# Agent 1: Researcher — equipped with web search capability
market_researcher = Agent(
    llm=llm,
    role="Market Researcher",
    goal="Analyze market trends for the product launch",
    backstory="Experienced in market trends and consumer behavior analysis",
    tools=[SerperDevTool()],  # Gives the agent live web search ability
    verbose=True,
)

# Agent 2: Strategist — synthesizes research into actionable strategy
strategist = Agent(
    llm=llm,
    role="Product Strategist",
    goal="Create effective positioning strategies for the product",
    backstory=f"Skilled in competitive positioning {strategist_backstory}",
    verbose=True,
)

## Step 4: Define the Tasks

Tasks define the work each agent performs. The second task implicitly depends on the first because CrewAI executes tasks sequentially by default — the strategist will receive the researcher's output as context.

In [ ]:
# Task 1: Researcher searches the web for market intelligence
gather_market_insights_task = Task(
    description=(
        f"Browse the internet to gather insights on current market trends "
        f"for the launch of the {product_name} product."
    ),
    expected_output=f"List of relevant market trends and consumer preferences, relevant to {product_name}",
    agent=market_researcher,
)

# Task 2: Strategist builds on the researcher's findings
develop_positioning_strategy_task = Task(
    description=(
        f"Based on the market insights, create a positioning strategy for "
        f"the {product_name} product, including analysis for impact and target audience."
    ),
    expected_output="A positioning strategy with target audience and impact notes",
    agent=strategist,
)

## Step 5: Assemble the Crew & Execute

The Crew connects agents to tasks. With `planning=True`, CrewAI will first generate an execution plan, then run each task in order. The researcher searches the web first, then the strategist uses those findings to craft a positioning strategy.

In [ ]:
crew = Crew(
    agents=[market_researcher, strategist],
    tasks=[gather_market_insights_task, develop_positioning_strategy_task],
    planning=True,  # Auto-generates an execution plan before running
    planning_llm=llm,  # Use our Groq LLM for planning too (defaults to OpenAI otherwise)
)

result = crew.kickoff()
print(result)